In [2]:
#!pip install -q pyarrow pandas matplotlib seaborn plotly scipy 

!pip install statsmodels nbformat>=4.2.0 --upgrade

In [4]:
import pyarrow.feather as feather
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from item_classifier import ItemClassifier, classify_items

In [3]:

table = feather.read_table('dataset/data_andre.feather', memory_map=True)
df= table.to_pandas()
df.head()

,date,item_id,value,cat_label,sdep_label,dep_label,dmn_label,promo_type_FRPG,promo_value_FRPG,promo_type_GAS,...,promo_value_DISC,promo_type_CIRC,promo_value_CIRC,promo_type_CIRE,promo_value_CIRE,promo_type_CLCP,promo_value_CLCP,promo_type_LFPE,promo_value_LFPE,store_id
0,2021-01-23,27,8,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0.0000,0,0.0,0,0.0,0,0.0,0,0.0,6269
1,2021-01-23,952568,6,carbonated sft drnks,pos subd grocery other,pos dept grocery,soda,0,0.0,0,...,0.0000,0,0.0,0,0.0,0,0.0,1,0.0,6269
2,2021-01-23,809,13,juices drnks shelf stbl,pos subd grocery other,pos dept grocery,juice/aseptic/new age,0,0.0,1,...,0.0000,0,0.0,0,0.0,0,0.0,0,0.0,6269
3,2021-01-23,20405,3,dairy chs,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0.1228,0,0.0,0,0.0,0,0.0,0,0.0,6269
4,2021-01-23,605573,5,refrigerated drnks,pos subd dairy other,pos dept dairy,cheese/yogurt/orange juice,0,0.0,0,...,0.0000,0,0.0,0,0.0,0,0.0,0,0.0,6269


In [ ]:


# Prepare data: ensure datetime index
df_indexed = df.copy()
if 'date' in df.columns:
    df_indexed['date'] = pd.to_datetime(df_indexed['date'])
    df_indexed = df_indexed.set_index('date')
elif 'timestamp' in df.columns:
    df_indexed['timestamp'] = pd.to_datetime(df_indexed['timestamp'])
    df_indexed = df_indexed.set_index('timestamp')



ModuleNotFoundError: No module named 'EDA'

In [8]:

# Classify all items
df_classified, summary = classify_items(
    df_indexed,
    seasonal_strength_threshold=0.5,    # Adjust based on your domain
    cv_threshold=0.8,                    # Adjust for intermittence detection
    initial_gap_fraction=0.2,            # 20% gap at start = new item
)

# Display summary
print("Label Distribution:")
print(summary['label_counts'])
print("\n" + "="*60)
print("Summary Statistics by Label:")
print(summary['summary_stats'])


Label Distribution:
{'intermittent': 1423, 'regular': 3, 'seasonal': 1}

Summary Statistics by Label:
             seasonal_strength                      cycle_cv            \
                          mean       min       max      mean       min   
label                                                                    
intermittent          0.195179  0.000000  0.487718  1.519094  0.829288   
regular               0.134505  0.093584  0.167897  0.790500  0.784011   
seasonal              0.500211  0.500211  0.500211  1.402622  1.402622   

                       initial_gap_fraction            
                   max                 mean  min  max  
label                                                  
intermittent  3.301381                  0.0  0.0  0.0  
regular       0.799755                  0.0  0.0  0.0  
seasonal      1.402622                  0.0  0.0  0.0  



## Classification Parameters Guide

The classifier uses three main parameters to categorize items:

### 1. `seasonal_strength_threshold` (default: 0.3)
- **What**: Strength of periodicity/seasonality in sales pattern (0-1 scale)
- **Higher values**: More strict seasonality detection
- **Lower values**: Easier to detect seasonal patterns
- **From document**: Items with sales/non-sales cycles that are regular and periodic

### 2. `cv_threshold` (default: 0.5)  
- **What**: Coefficient of Variation in cycle intervals
- **Higher values**: Allow more variability in intervals (more "erratic")
- **Lower values**: Stricter intermittence detection
- **From document**: Intermittent items have intervals that "don't share any common pattern"

### 3. `initial_gap_fraction` (default: 0.2)
- **What**: Fraction of zero-sales at the beginning (0-1 scale)
- **Higher values**: Allow longer initial gaps
- **Lower values**: Stricter new item detection
- **From document**: New items have "meaningful non-sales cycle at the beginning"

## Tuning Tips
- Start with defaults, then adjust based on your visualization results
- If too many items are "seasonal" → increase `seasonal_strength_threshold`
- If too many items are "intermittent" → increase `cv_threshold`
- If too many items are "new" → increase `initial_gap_fraction`


In [7]:
# Import visualization libraries
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Create subplots: bar chart and pie chart
label_counts = summary['label_counts']

fig = make_subplots(
    rows=1, cols=2,
    specs=[[{"type": "bar"}, {"type": "pie"}]],
    subplot_titles=("Distribution Count", "Distribution Percentage")
)

# Add bar chart
fig.add_trace(
    go.Bar(x=list(label_counts.keys()), y=list(label_counts.values()), 
           marker=dict(color=['#2ecc71', '#3498db', '#e74c3c', '#f39c12'][:len(label_counts)]),
           text=list(label_counts.values()), textposition='outside',
           name='Count'),
    row=1, col=1
)

# Add pie chart
fig.add_trace(
    go.Pie(labels=list(label_counts.keys()), values=list(label_counts.values()),
           marker=dict(colors=['#2ecc71', '#3498db', '#e74c3c', '#f39c12'][:len(label_counts)]),
           name='Percentage'),
    row=1, col=2
)

# Update layout
fig.update_xaxes(title_text="Classification Label", row=1, col=1)
fig.update_yaxes(title_text="Count", row=1, col=1)
fig.update_layout(height=500, showlegend=False, title_text="Item Classification Distribution")

fig.show()

# Print detailed breakdown
print("\nDetailed Classification Distribution:")
print(f"Total items: {sum(label_counts.values())}")
print("\nBreakdown:")
for label, count in label_counts.items():
    percentage = (count / sum(label_counts.values())) * 100
    print(f"  {label:12s}: {count:6d} items ({percentage:5.1f}%)")


Detailed Classification Distribution:
Total items: 1427

Breakdown:
  intermittent:   1314 items ( 92.1%)
  seasonal    :    113 items (  7.9%)


In [10]:
# Plot time series for each classification category
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Get sample items from each label
labels_to_plot = ['regular', 'seasonal', 'intermittent']
n_samples_per_label = 1

fig = make_subplots(
    rows=3, cols=1,
    subplot_titles=[f"{label.upper()} Items (Sample)" for label in labels_to_plot],
    specs=[[{"secondary_y": False}], [{"secondary_y": False}], [{"secondary_y": False}]],
    vertical_spacing=0.12
)

colors = {'regular': '#2ecc71', 'seasonal': '#3498db', 'intermittent': '#e74c3c'}

for row_idx, label in enumerate(labels_to_plot, 1):
    # Get items with this label
    items_with_label = df_classified[df_classified['item_label'] == label]['item_id'].unique()
    
    if len(items_with_label) == 0:
        print(f"No items found with label: {label}")
        continue
    
    # Plot first n_samples_per_label items
    for i, item_id in enumerate(items_with_label[:n_samples_per_label]):
        item_data = df_classified[df_classified['item_id'] == item_id].sort_index()
        
        fig.add_trace(
            go.Scatter(
                x=item_data.index,
                y=item_data['value'],
                mode='lines+markers',
                name=f"{label.capitalize()} Item {i+1}",
                line=dict(color=colors[label], width=2),
                opacity=0.7
            ),
            row=row_idx, col=1
        )

# Update layout
fig.update_xaxes(title_text="Date", row=3, col=1)
fig.update_yaxes(title_text="Sales Value", row=1, col=1)
fig.update_yaxes(title_text="Sales Value", row=2, col=1)
fig.update_yaxes(title_text="Sales Value", row=3, col=1)

fig.update_layout(
    height=900,
    title_text="Time Series of Items by Classification Category",
    hovermode='x unified',
    showlegend=True
)

fig.show()

In [ ]:
def analyze_product_timeseries(item_id):
    """
    Analyze and visualize the time series for a specific product.
    
    Parameters:
    item_id : int or str - The product ID to analyze
    """
    # Get the product data
    product_data = df_classified[df_classified['item_id'] == item_id].sort_index()
    
    if len(product_data) == 0:
        print(f"⚠️ Item ID {item_id} not found in dataset")
        return
    
    # Get classification info
    label = product_data['item_label'].iloc[0]
    dep = product_data['dep_label'].iloc[0] if 'dep_label' in product_data.columns else "N/A"
    
    # Display product info
    print(f"\n{'='*70}")
    print(f"Product ID: {item_id}")
    print(f"Department: {dep}")
    print(f"Classification: {label.upper()}")
    print(f"{'='*70}")
    
    # Statistics
    print(f"\nTime Series Statistics:")
    print(f"  Period: {product_data.index.min().date()} to {product_data.index.max().date()}")
    print(f"  Total records: {len(product_data)}")
    print(f"  Non-zero sales: {(product_data['value'] > 0).sum()}")
    print(f"  Zero-sales periods: {(product_data['value'] == 0).sum()}")
    print(f"\n  Sales Stats:")
    print(f"    Mean: {product_data['value'].mean():.2f}")
    print(f"    Median: {product_data['value'].median():.2f}")
    print(f"    Min: {product_data['value'].min():.2f}")
    print(f"    Max: {product_data['value'].max():.2f}")
    print(f"    Std Dev: {product_data['value'].std():.2f}")
    
    # Plot time series
    fig = go.Figure()
    
    fig.add_trace(
        go.Scatter(
            x=product_data.index,
            y=product_data['value'],
            mode='lines+markers',
            name=f'Item {item_id}',
            line=dict(color='#3498db', width=2),
            marker=dict(size=6)
        )
    )
    
    # Color code zero vs non-zero sales
    zero_sales = product_data[product_data['value'] == 0]
    if len(zero_sales) > 0:
        fig.add_trace(
            go.Scatter(
                x=zero_sales.index,
                y=zero_sales['value'],
                mode='markers',
                name='Zero Sales',
                marker=dict(color='red', size=8, symbol='x'),
                showlegend=True
            )
        )
    
    fig.update_layout(
        title=f"Time Series: Item {item_id} ({label.upper()})",
        xaxis_title="Date",
        yaxis_title="Sales Value",
        height=500,
        hovermode='x unified',
        template='plotly_white'
    )
    
    fig.show()

In [ ]:
def search_products(label=None, department=None, item_id=None):
    """
    Search for products by classification label, department, or item ID.
    
    Parameters:
    label : str - Filter by classification ('regular', 'seasonal', 'intermittent', 'new')
    department : str - Filter by department name
    item_id : int - Exact item ID to find
    
    Returns:
    DataFrame with matching products
    """
    result = df_classified.copy()
    
    if item_id is not None:
        result = result[result['item_id'] == item_id]
    
    if label is not None:
        result = result[result['item_label'] == label.lower()]
    
    if department is not None:
        result = result[result['dep_label'].str.contains(department, case=False, na=False)]
    
    # Get unique items
    unique_items = result[['item_id', 'item_label', 'dep_label']].drop_duplicates()
    
    print(f"Found {len(unique_items)} items matching criteria:")
    print(f"\nClassification breakdown:")
    print(unique_items['item_label'].value_counts())
    
    return unique_items
